# 09 - 自定义数据集**学习目标：**- 学会准备自己的数据集- 了解常用标注工具- 掌握标注格式转换（COCO/VOC → YOLO）- 学会划分训练集/验证集/测试集---

## 1. 准备数据集的流程```1. 收集图片   ↓2. 标注图片（画框 + 分类）   ↓3. 格式转换（转为 YOLO 格式）   ↓4. 划分数据集（train/val/test）   ↓5. 创建 data.yaml   ↓6. 开始训练```

## 2. 常用标注工具| 工具 | 特点 | 链接 ||------|------|------|| **LabelImg** | 最简单，桌面应用 | github.com/heartexlabs/labelImg || **CVAT** | 功能强大，支持团队协作 | cvat.ai || **Roboflow** | 云端标注 + 数据管理 | roboflow.com || **Label Studio** | 开源，支持多种标注类型 | labelstud.io |**LabelImg** 最适合入门：```bashpip install labelimglabelimg```选择 YOLO 格式保存即可！

## 3. 创建目录结构标准 YOLO 数据集结构：

In [ ]:
from pathlib import Path# 创建示例数据集结构dataset_dir = Path("../data/my_dataset")dirs = [    dataset_dir / "images" / "train",    dataset_dir / "images" / "val",    dataset_dir / "images" / "test",    dataset_dir / "labels" / "train",    dataset_dir / "labels" / "val",    dataset_dir / "labels" / "test",]for d in dirs:    d.mkdir(parents=True, exist_ok=True)print("数据集目录结构已创建：")for d in dirs:    print(f"  {d.relative_to(Path('..'))}")

## 4. 标注格式转换### 从 Pascal VOC 转换为 YOLOVOC 格式 (XML)：```xml<annotation>    <size>        <width>640</width>        <height>480</height>    </size>    <object>        <name>cat</name>        <bndbox>            <xmin>100</xmin>            <ymin>50</ymin>            <xmax>400</xmax>            <ymax>350</ymax>        </bndbox>    </object></annotation>```YOLO 格式 (TXT)：```0 0.390625 0.416667 0.46875 0.625```

In [ ]:
import xml.etree.ElementTree as ETfrom pathlib import Pathdef voc_to_yolo(xml_path, class_names):    """将 Pascal VOC XML 标注转换为 YOLO TXT 格式.        Args:        xml_path: VOC XML 文件路径        class_names: 类别名称列表，如 ['cat', 'dog', 'bird']        Returns:        list of YOLO 格式字符串    """    tree = ET.parse(xml_path)    root = tree.getroot()        # 获取图片尺寸    size = root.find('size')    img_w = int(size.find('width').text)    img_h = int(size.find('height').text)        yolo_lines = []    for obj in root.findall('object'):        name = obj.find('name').text        if name not in class_names:            continue                class_id = class_names.index(name)        bbox = obj.find('bndbox')        xmin = float(bbox.find('xmin').text)        ymin = float(bbox.find('ymin').text)        xmax = float(bbox.find('xmax').text)        ymax = float(bbox.find('ymax').text)                # 转换为 YOLO 格式        cx = ((xmin + xmax) / 2) / img_w        cy = ((ymin + ymax) / 2) / img_h        w = (xmax - xmin) / img_w        h = (ymax - ymin) / img_h                yolo_lines.append(f"{class_id} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")        return yolo_lines# 示例print("转换示例：")print("VOC: xmin=100, ymin=50, xmax=400, ymax=350, img=640×480")print("YOLO: 0 0.390625 0.416667 0.46875 0.625")

### 从 COCO JSON 转换为 YOLO

In [ ]:
import jsondef coco_to_yolo(json_path, output_dir, class_mapping=None):    """将 COCO JSON 标注转换为 YOLO TXT 格式.        Args:        json_path: COCO annotation JSON 文件路径        output_dir: 输出目录        class_mapping: 可选的类别映射 {coco_id: yolo_id}    """    with open(json_path) as f:        data = json.load(f)        # 建立 image_id → (width, height, filename) 映射    images = {img['id']: img for img in data['images']}        # 按图片分组标注    from collections import defaultdict    img_annotations = defaultdict(list)    for ann in data['annotations']:        img_annotations[ann['image_id']].append(ann)        output_dir = Path(output_dir)    output_dir.mkdir(parents=True, exist_ok=True)        for img_id, anns in img_annotations.items():        img_info = images[img_id]        img_w, img_h = img_info['width'], img_info['height']                lines = []        for ann in anns:            x, y, w, h = ann['bbox']  # COCO 格式: x,y 是左上角                        # 转换为 YOLO 格式            cx = (x + w/2) / img_w            cy = (y + h/2) / img_h            nw = w / img_w            nh = h / img_h                        cat_id = ann['category_id']            if class_mapping:                cat_id = class_mapping.get(cat_id, cat_id)                        lines.append(f"{cat_id} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")                # 写入文件        filename = Path(img_info['file_name']).stem + '.txt'        with open(output_dir / filename, 'w') as f:            f.write('\n'.join(lines))        print(f"转换完成，{len(img_annotations)} 个标注文件保存到 {output_dir}")print("COCO → YOLO 转换函数已定义")

## 5. 划分数据集

In [ ]:
import random
import shutil
from pathlib import Path

def split_dataset(images_dir, labels_dir, output_dir, 
                  train_ratio=0.7, val_ratio=0.2, test_ratio=0.1,
                  seed=42):
    """划分数据集为 train/val/test."""
    random.seed(seed)
    
    images_dir = Path(images_dir)
    labels_dir = Path(labels_dir)
    output_dir = Path(output_dir)
    
    # 获取所有图片
    img_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        img_files.extend(images_dir.glob(ext))
    img_files = sorted(img_files)
    random.shuffle(img_files)
    
    n = len(img_files)
    n_train = int(n * train_ratio)
    n_val = int(n * val_ratio)
    
    splits = {
        'train': img_files[:n_train],
        'val': img_files[n_train:n_train+n_val],
        'test': img_files[n_train+n_val:],
    }
    
    for split_name, files in splits.items():
        (output_dir / 'images' / split_name).mkdir(parents=True, exist_ok=True)
        (output_dir / 'labels' / split_name).mkdir(parents=True, exist_ok=True)
        
        for img_path in files:
            shutil.copy(img_path, output_dir / 'images' / split_name / img_path.name)
            label_path = labels_dir / (img_path.stem + '.txt')
            if label_path.exists():
                shutil.copy(label_path, output_dir / 'labels' / split_name / label_path.name)
        
        print(f"  {split_name}: {len(files)} 张图片")
    
    print(f"\n划分完成！保存到: {output_dir}")

# 使用示例（取消注释运行）
# split_dataset(
#     images_dir="../data/my_dataset/images/all",
#     labels_dir="../data/my_dataset/labels/all",
#     output_dir="../data/my_dataset",
# )
print("划分函数已定义，准备好数据后取消注释运行")

## 6. 创建 data.yaml

In [ ]:
import yaml
from pathlib import Path

def create_data_yaml(dataset_dir, class_names, train='images/train', val='images/val', test=None):
    """创建 YOLO 训练所需的 data.yaml."""
    config = {
        'path': str(Path(dataset_dir).resolve()),
        'train': train,
        'val': val,
        'nc': len(class_names),
        'names': {i: name for i, name in enumerate(class_names)},
    }
    if test:
        config['test'] = test
    
    yaml_path = Path(dataset_dir) / 'data.yaml'
    with open(yaml_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, allow_unicode=True)
    print(f"data.yaml 已创建: {yaml_path}")
    return yaml_path

# 示例
create_data_yaml(
    dataset_dir="../data/my_dataset",
    class_names=['cat', 'dog', 'bird'],
)

## 7. 数据集质量检查在训练前，务必检查数据集质量：

In [ ]:
from pathlib import Path
from collections import Counter

def check_dataset(dataset_dir):
    """检查数据集质量"""
    dataset_dir = Path(dataset_dir)
    issues = []
    
    for split in ['train', 'val']:
        img_dir = dataset_dir / 'images' / split
        lbl_dir = dataset_dir / 'labels' / split
        
        if not img_dir.exists():
            issues.append(f"缺少目录: {img_dir}")
            continue
        
        imgs = {f.stem for f in img_dir.glob('*.jpg')}
        lbls = {f.stem for f in lbl_dir.glob('*.txt')}
        
        missing_labels = imgs - lbls
        missing_images = lbls - imgs
        
        if missing_labels:
            issues.append(f"{split}: {len(missing_labels)} 张图片没有标注")
        if missing_images:
            issues.append(f"{split}: {len(missing_images)} 个标注没有对应图片")
        
        class_counts = Counter()
        for lbl_file in lbl_dir.glob('*.txt'):
            with open(lbl_file) as f:
                for line in f:
                    if line.strip():
                        class_counts[int(line.split()[0])] += 1
        
        print(f"\n{split} 集:")
        print(f"  图片数: {len(imgs)}")
        print(f"  标注数: {len(lbls)}")
        print(f"  标注框总数: {sum(class_counts.values())}")
    
    if issues:
        print("\n⚠️ 发现问题:")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("\n✅ 数据集检查通过！")

# 检查 COCO128
check_dataset("../data/coco128")

## 📝 学习检查

加载本章检查点和练习，检验学习效果：

```python
from yolo_learn.pedagogy.checkpoint import Quiz
from yolo_learn.pedagogy.scaffold import ExerciseSet
from yolo_learn.pedagogy.reflection import ReflectionSet, MistakeLibrary
```


In [ ]:
# 加载本章检查点
quiz = Quiz.from_yaml("09_custom_dataset")
print(f"本章 {quiz.total} 个检查点 + 预测练习")

# 查看第一题
print(quiz.ask(0))


### ✍️ 练习


In [ ]:
# 加载本章练习
exercises = ExerciseSet.from_yaml("09_custom_dataset")
print(f"本章 {exercises.total} 个练习")
for ex in exercises.exercises:
    print()
    print(ex.show())


### 🤔 反思与自评


In [ ]:
# 加载反思问题和自评量表
reflections = ReflectionSet.from_yaml("09_custom_dataset")
print(reflections.show_reflections())
print()
print(reflections.show_assessments())

# 常见错误
mistakes = MistakeLibrary()
print()
print(mistakes.show_for_topic("custom_dataset"))


## 8. 练习### 练习 1: 准备一个 3 类数据集选择 3 种物体（如猫、狗、鸟），收集 50 张图片并用 LabelImg 标注。### 练习 2: 格式转换如果你有 VOC 或 COCO 格式的数据，用上面的函数转换为 YOLO 格式。### 练习 3: 训练自定义模型用你准备的数据集训练一个模型，观察效果。---**上一节**: [08 - 数据增强](08_data_augmentation.ipynb)**下一节**: [10 - 导出与部署](10_export_and_deploy.ipynb) - 模型导出和优化